# 使用 time travel

## 概览

LangGraph 通过 [checkpoints](https://docs.langchain.com/oss/langgraph/checkpointers#checkpoints) 支持 time travel:

- **[重放](#replay)**: 从先前的 checkpoint 重试。
- **[分叉](#fork)**: 从带修改后 state 的先前 checkpoint 分叉, 以探索替代路径。

两者的运作方式都是从先前的 checkpoint 恢复。checkpoint 之前的 node 不会重新执行 (结果已经保存)。checkpoint 之后的 node 会重新执行, 包括任何 LLM 调用、API 请求与 [interrupts](https://docs.langchain.com/oss/langgraph/interrupts) (它们可能产生不同的结果)。

## 重放

用先前 checkpoint 的 config 调用 graph, 即可从该点重放。

> **警告**
>
> 重放会重新执行 node — 它不只是从缓存读取。LLM 调用、API 请求与 [interrupts](https://docs.langchain.com/oss/langgraph/interrupts) 会再次触发, 并可能返回不同的结果。从最终 checkpoint (没有 `next` node) 重放是一个空操作。

![Replay](https://docs.langchain.com/oss/images/re_play.png)

使用 `get_state_history` 找到你想从中重放的 checkpoint, 然后用该 checkpoint 的 config 调用 `invoke`:

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
from typing_extensions import TypedDict, NotRequired

from langchain_core.utils.uuid import uuid7
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"}


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .compile(checkpointer=checkpointer)
)

# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid7())}}
result = graph.invoke({}, config)

# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config))
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

# Step 3: Replay from a specific checkpoint
# Find the checkpoint before write_joke
before_joke = next(s for s in history if s.next == ("write_joke",))
replay_result = graph.invoke(None, before_joke.config)
# write_joke re-executes (runs again), generate_topic does not

next=(), checkpoint_id=1f1bdb60-6dbd-6b7d-8002-942a05588164
next=('write_joke',), checkpoint_id=1f1bdb60-6dba-6acd-8001-a9937904768b
next=('generate_topic',), checkpoint_id=1f1bdb60-6db6-665b-8000-87e2163c4dae
next=('__start__',), checkpoint_id=1f1bdb60-6db1-6e26-bfff-2df0c905bf7d


## 分叉

分叉会从带修改后 state 的一个过去 checkpoint 创建新分支。在先前 checkpoint 上调用 `update_state` 来创建该分叉, 然后用 `None` 调用 `invoke` 以继续执行。

![Fork](https://docs.langchain.com/oss/images/checkpoints_full_story.jpg)

> **警告**
>
> `update_state` **不会**回滚 thread。它会创建一个从指定点分出的新 checkpoint。原始执行历史保持完整。

In [2]:
# Find checkpoint before write_joke
history = list(graph.get_state_history(config))
before_joke = next(s for s in history if s.next == ("write_joke",))

# Fork: update state to change the topic
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
)

# Resume from the fork — write_joke re-executes with the new topic
fork_result = graph.invoke(None, fork_config)
print(fork_result["joke"])  # A joke about chickens, not socks

Why do chickens disappear? They elope!


### 从特定 node 分叉

当你调用 `update_state` 时, 值会使用指定 node 的 writer 来应用 (包括 [reducers](https://docs.langchain.com/oss/langgraph/graph-api#reducers))。checkpoint 会记录是该 node 产生了该更新, 执行则从该 node 的后继节点恢复。

默认情况下, LangGraph 会从 checkpoint 的版本历史推断 `as_node`。当从特定 checkpoint 分叉时, 这种推断几乎总是正确的。

以下情况请显式指定 `as_node`:

- **并行分支**: 多个 node 在同一个步骤中更新了 state, 而 LangGraph 无法确定哪个是最后一个 (`InvalidUpdateError`)。
- **没有执行历史**: 在一个全新的 thread 上设置 state (常见于 [测试](https://docs.langchain.com/oss/langgraph/test))。
- **跳过 node**: 把 `as_node` 设为一个更靠后的 node, 让 graph 认为该 node 已经运行过。

In [3]:
# graph: generate_topic -> write_joke

# Treat this update as if generate_topic produced it.
# Execution resumes at write_joke (the successor of generate_topic).
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
    as_node="generate_topic",
)

## Interrupts

如果你的 graph 使用 `interrupt` 来实现 [human-in-the-loop](https://docs.langchain.com/oss/langgraph/interrupts) workflow, 那么在 time travel 期间 interrupts 总会被重新触发。包含该 interrupt 的 node 会重新执行, `interrupt()` 会暂停以等待一个新的 `Command(resume=...)`。

In [4]:
class State(TypedDict):
    value: list[str]

def ask_human(state: State):
    answer = interrupt("What is your name?")
    return {"value": [f"Hello, {answer}!"]}

def final_step(state: State):
    return {"value": ["Done"]}

graph = (
    StateGraph(State)
    .add_node("ask_human", ask_human)
    .add_node("final_step", final_step)
    .add_edge(START, "ask_human")
    .add_edge("ask_human", "final_step")
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# First run: hits interrupt
graph.invoke({"value": []}, config)
# Resume with answer
graph.invoke(Command(resume="Alice"), config)

# Replay from before ask_human
history = list(graph.get_state_history(config))
before_ask = [s for s in history if s.next == ("ask_human",)][-1]

replay_result = graph.invoke(None, before_ask.config)
# Pauses at interrupt — waiting for new Command(resume=...)

# Fork from before ask_human
fork_config = graph.update_state(before_ask.config, {"value": ["forked"]})
fork_result = graph.invoke(None, fork_config)
# Pauses at interrupt — waiting for new Command(resume=...)

# Resume the forked interrupt with a different answer
graph.invoke(Command(resume="Bob"), fork_config)
# Result: {"value": ["forked", "Hello, Bob!", "Done"]}

{'value': ['Done']}

### 多个 interrupts

如果你的 graph 会在多个点收集输入 (例如一个多步骤表单), 你可以从 interrupts 之间分叉, 从而在不重新询问先前问题的情况下更改之后的答案。

```python
def ask_name(state):
    name = interrupt("What is your name?")
    return {"value": [f"name:{name}"]}

def ask_age(state):
    age = interrupt("How old are you?")
    return {"value": [f"age:{age}"]}

# Graph: ask_name -> ask_age -> final
# After completing both interrupts:

# Fork from BETWEEN the two interrupts (after ask_name, before ask_age)
history = list(graph.get_state_history(config))
between = [s for s in history if s.next == ("ask_age",)][-1]

fork_config = graph.update_state(between.config, {"value": ["modified"]})
result = graph.invoke(None, fork_config)
# ask_name result preserved ("name:Alice")
# ask_age pauses at interrupt — waiting for new answer
```

## Subgraphs

对 [subgraphs](https://docs.langchain.com/oss/langgraph/use-subgraphs) 进行 time travel 取决于该 subgraph 是否拥有自己的 checkpointer。这决定了你能从哪个粒度的 checkpoint 进行 time travel。

## 继承的 checkpointer (默认)

默认情况下, subgraph 会继承父级的 checkpointer。父级会把整个 subgraph 视为**单个 super-step** — 整个 subgraph 执行只有一个父级 checkpoint。从 subgraph 之前进行 time travel 会从头重新执行它。

你无法 time travel 到默认 subgraph 中 node *之间*的某个点 — 你只能从父级进行 time travel。

```python
# Subgraph without its own checkpointer (default)
subgraph = (
    StateGraph(State)
    .add_node("step_a", step_a)       # Has interrupt()
    .add_node("step_b", step_b)       # Has interrupt()
    .add_edge(START, "step_a")
    .add_edge("step_a", "step_b")
    .compile()  # No checkpointer — inherits from parent
)

graph = (
    StateGraph(State)
    .add_node("subgraph_node", subgraph)
    .add_edge(START, "subgraph_node")
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Complete both interrupts
graph.invoke({"value": []}, config)            # Hits step_a interrupt
graph.invoke(Command(resume="Alice"), config)  # Hits step_b interrupt
graph.invoke(Command(resume="30"), config)     # Completes

# Time travel from before the subgraph
history = list(graph.get_state_history(config))
before_sub = [s for s in history if s.next == ("subgraph_node",)][-1]

fork_config = graph.update_state(before_sub.config, {"value": ["forked"]})
result = graph.invoke(None, fork_config)
# The entire subgraph re-executes from scratch
# You cannot time travel to a point between step_a and step_b
```

## Subgraph checkpointer

在 subgraph 上设置 `checkpointer=True`, 让它拥有自己的 checkpoint 历史。这会在 subgraph **内部**的每一步创建 checkpoint, 让你能从其内部的某个特定点进行 time travel — 例如两个 interrupts 之间。

使用带 `subgraphs=True` 的 `get_state` 访问 subgraph 自己的 checkpoint config, 然后从中分叉:

```python
# Subgraph with its own checkpointer
subgraph = (
    StateGraph(State)
    .add_node("step_a", step_a)       # Has interrupt()
    .add_node("step_b", step_b)       # Has interrupt()
    .add_edge(START, "step_a")
    .add_edge("step_a", "step_b")
    .compile(checkpointer=True)  # Own checkpoint history
)

graph = (
    StateGraph(State)
    .add_node("subgraph_node", subgraph)
    .add_edge(START, "subgraph_node")
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Run until step_a interrupt
graph.invoke({"value": []}, config)

# Resume step_a -> hits step_b interrupt
graph.invoke(Command(resume="Alice"), config)

# Get the subgraph's own checkpoint (between step_a and step_b)
parent_state = graph.get_state(config, subgraphs=True)
sub_config = parent_state.tasks[0].state.config

# Fork from the subgraph checkpoint
fork_config = graph.update_state(sub_config, {"value": ["forked"]})
result = graph.invoke(None, fork_config)
# step_b re-executes, step_a's result is preserved
```

关于配置 subgraph checkpointers 的更多内容, 见 [subgraph persistence](https://docs.langchain.com/oss/langgraph/use-subgraphs#subgraph-persistence)。